# Tutorial de Razonamiento con Owlready2 con `GameOntologyv3.owl`

**Objetivo.** Aprender a utilizar el razonador para observar **inferencias** y posibles **inconsistencias** en una ontología OWL:

1. Cargar la ontología.
7. Ejecutar un razonador.
8. Observar **inferencias** y posibles **inconsistencias**.

La ontología de partida para este tutorial es `GameOntology_ampliada.owl`, que se guardó como resultado de la ejecución del tutorial anterior.


In [1]:
#Para comprobar que se tiene Java instalado, necesario para ejecutar el razonador
import subprocess
subprocess.run(["java", "-version"])

CompletedProcess(args=['java', '-version'], returncode=0)

## 1. Cargar la ontología

In [1]:
from owlready2 import *

onto = get_ontology("GameOntology_ampliada.owl")

try:
    onto.load()
    print("Ontología cargada correctamente")
except FileNotFoundError:
    print("ERROR: no se encuentra el fichero")
except Exception as e:
    print("ERROR al cargar la ontología:")
    print(type(e).__name__, "-", e)

Ontología cargada correctamente


In [2]:
print("Número de clases:", len(list(onto.classes())))
print("Número de propiedades de objeto:", len(list(onto.object_properties())))
print("Número de propiedades de datos:", len(list(onto.data_properties())))
print("Número de individuos:", len(list(onto.individuals())))
print("Individuos definidos:", list(onto.individuals()))

Número de clases: 42
Número de propiedades de objeto: 37
Número de propiedades de datos: 6
Número de individuos: 9
Individuos definidos: [GameOntology_ampliada.juego1, GameOntology_ampliada.heroina, GameOntology_ampliada.villano, GameOntology_ampliada.juego2, GameOntology_ampliada.ana, GameOntology_ampliada.id_001, GameOntology_ampliada.luis, GameOntology_ampliada.marta, GameOntology_ampliada.mario]


## 2. Preparar inferencias para el razonador

Recordemos que la ontología contiene:

- `ana` y `luis` son `Player`.
- ambos tienen una relación `playsGame` con un `Game`.
- `JugadorActivo` está definido mediante una condición necesaria y suficiente.

Por tanto, esperamos que el razonador pueda clasificarlos como `JugadorActivo`.

Además, tenemos la propiedad `outranks`, que es transitiva. Por tanto, si `ana` outranks a `luis`, y `luis` outranks a `marta`, el razonador debería inferir que `ana` outranks a `marta`.

In [3]:
print("Tipos de Ana ANTES del razonador:", onto.ana.is_a)
print("Tipos de Luis ANTES del razonador:", onto.luis.is_a)

print("Ana outranks (valores explícitos):", onto.ana.outranks)
print("Ana outranks (valores indirectos):", onto.ana.INDIRECT_outranks)

print("Luis outranks:", onto.luis.outranks)
print("Marta outranks:", onto.marta.outranks)

print("\n¿Marta aparece directamente entre los valores de outranks de Ana?")
print(onto.marta in onto.ana.outranks)

Tipos de Ana ANTES del razonador: [GameOntology_ampliada.Player]
Tipos de Luis ANTES del razonador: [GameOntology_ampliada.Player]
Ana outranks (valores explícitos): [GameOntology_ampliada.luis]
Ana outranks (valores indirectos): [GameOntology_ampliada.marta, GameOntology_ampliada.luis]
Luis outranks: [GameOntology_ampliada.marta]
Marta outranks: []

¿Marta aparece directamente entre los valores de outranks de Ana?
False


## 3. Ejecutar el razonador

Owlready2 puede invocar razonadores externos. Aquí usamos **Pellet**.

`infer_property_values=True` pide materializar también valores de propiedades inferidos.

> Pellet necesita Java. Si el entorno no dispone de Java, esta celda puede producir un error aunque la ontología esté correctamente cargada.


In [4]:
try:
    with onto:
        sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)
    print("Razonamiento completado")
except OwlReadyInconsistentOntologyError:
    print("La ontología es INCONSISTENTE")
except Exception as e:
    print("No se pudo ejecutar Pellet:")
    print(type(e).__name__, "-", e)

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

* Owlready * Adding relation GameOntology_ampliada.ana outranks GameOntology_ampliada.marta
Razonamiento completado


* Owlready2 * Pellet took 0.9974336624145508 seconds
* Owlready * Reparenting GameOntology_ampliada.ana: {GameOntology_ampliada.Player} => {GameOntology_ampliada.JugadorActivo}
* Owlready * Reparenting GameOntology_ampliada.luis: {GameOntology_ampliada.Player} => {GameOntology_ampliada.JugadorActivo}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


### Comprobar las inferencias

In [5]:
print("Tipos de Ana DESPUÉS:", onto.ana.is_a)
print("Tipos de Luis DESPUÉS:", onto.luis.is_a)

print("\n¿Ana es JugadorActivo?")
print(onto.JugadorActivo in onto.ana.is_a)

print("\nJugadores por debajo de Ana según outranks:")
print(onto.ana.outranks)

Tipos de Ana DESPUÉS: [GameOntology_ampliada.JugadorActivo]
Tipos de Luis DESPUÉS: [GameOntology_ampliada.JugadorActivo]

¿Ana es JugadorActivo?
True

Jugadores por debajo de Ana según outranks:
[GameOntology_ampliada.luis, GameOntology_ampliada.marta]


## 4. Experimento: propiedad inversamente funcional

Creamos otro nombre de individuo, `jugadorX`, y le asignamos **el mismo identificador** que a Ana.

Como `hasPlayerIdentifier` es inversamente funcional:

**hasPlayerIdentifier(ana, id_001)**  
**hasPlayerIdentifier(jugadorX, id_001)**

implica semánticamente que `ana` y `jugadorX` deben representar al mismo individuo.

> OWL no adopta la *Unique Name Assumption*: dos nombres diferentes no significan necesariamente dos individuos diferentes.


In [6]:
jugadorX = onto.Player("jugadorX")
jugadorX.hasPlayerIdentifier.append(onto.id_001)

# Volvemos a ejecutar el razonador
try:
    with onto:
        sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)
    print("Razonamiento completado")
except OwlReadyInconsistentOntologyError:
    print("La ontología es INCONSISTENTE")
except Exception as e:
    print(type(e).__name__, "-", e)

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

* Owlready * Adding relation GameOntology_ampliada.jugadorX playsGame GameOntology_ampliada.juego1
* Owlready * Adding relation GameOntology_ampliada.jugadorX outranks GameOntology_ampliada.marta
* Owlready * Adding relation GameOntology_ampliada.jugadorX outranks GameOntology_ampliada.luis
* Owlready * Adding relation GameOntology_ampliada.jugadorX isTeammateOf GameOntology_ampliada.luis
* Owlready * Adding relation GameOntology_ampliada.luis isTeammateOf GameOntology_ampliada.jugadorX
* Owlready * Adding relation GameOntology_ampliada.jugadorX username AnaPlayer
Razonamiento completado


* Owlready2 * Pellet took 0.9756555557250977 seconds
* Owlready * Reparenting GameOntology_ampliada.jugadorX: {GameOntology_ampliada.Player} => {GameOntology_ampliada.JugadorActivo}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


Como `hasPlayerIdentifier` es inversamente funcional, el razonador puede inferir que `ana` y `jugadorX` denotan al mismo individuo. Esto no implica necesariamente que Owlready2 elimine uno de los nombres o que ambos objetos Python se fusionen físicamente.

In [8]:
print(jugadorX.username)


['AnaPlayer']


Vamos a crear un nuevo individuo `jugadorY` con el mismo identificador que `ana` y vamos a indicar que son individuos diferentes. Al ejecutar de nuevo el razonador, debería detectar una inconsistencia.

In [10]:
jugadorY = onto.Player("jugadorY")
jugadorY.hasPlayerIdentifier.append(onto.id_001)

AllDifferent([onto.ana, jugadorY])

# Volvemos a ejecutar el razonador
try:
    with onto:
        sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)
    print("Razonamiento completado")
except OwlReadyInconsistentOntologyError:
    print("La ontología es INCONSISTENTE")

    # Obtener la explicación: usamos Pellet que sí nos devuelve una explicación
    try:
        sync_reasoner_pellet(debug=2)
    except Exception as e:
        text = str(e)
        print(text[text.index('Explanation'):])

except Exception as e:
    print(type(e).__name__, "-", e)




* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

La ontología es INCONSISTENTE


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

Explanation(s): 
1)   ana hasPlayerIdentifier id_001
     InverseFunctional hasPlayerIdentifier
     jugadorY hasPlayerIdentifier id_001
     ana differentFrom jugadorY






In [11]:
#Eliminamos jugadorY para poder seguir trabajando con la ontología
destroy_entity(jugadorY)

## 5. Experimento: clases disjuntas

Hemos declarado:

**JugadorCasual ⊓ JugadorCompetitivo ⊑ ⊥**

Por tanto, ningún individuo puede pertenecer a ambas clases.

In [12]:
onto.marta.is_a.append(onto.JugadorCasual)
onto.marta.is_a.append(onto.JugadorCompetitivo)

try:
    with onto:
        sync_reasoner_pellet()
    print("La ontología es consistente")
except OwlReadyInconsistentOntologyError:
    print("La ontología es INCONSISTENTE")

    # Obtener la explicación: usamos Pellet que sí nos devuelve una explicación
    try:
        sync_reasoner_pellet(debug=2)
    except Exception as e:
        text = str(e)
        print(text[text.index('Explanation'):])

except Exception as e:
    print(type(e).__name__, "-", e)

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

La ontología es INCONSISTENTE


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

Explanation(s): 
1)   JugadorCasual disjointWith JugadorCompetitivo
     marta type JugadorCasual
     marta type JugadorCompetitivo






### ¿Por qué es inconsistente?

Porque tendríamos simultáneamente **JugadorCasual(marta)** y **JugadorCompetitivo(marta)** y hemos declarado **JugadorCasual ⊓ JugadorCompetitivo ⊑ ⊥**.
Por tanto, el modelo exigiría que Marta perteneciera a una clase imposible (⊥).
Restauramos la ontología para continuar trabajando en un estado consistente.


In [13]:
#Dejamos a Marta SOLO como JugadorCompetitivo
onto.marta.is_a = [onto.JugadorCompetitivo]

## 6. Guardar la ontología ampliada

Las clases, propiedades e individuos que hemos creado pueden guardarse en un nuevo fichero OWL.


In [26]:
onto.save(file="GameOntology_ampliada.owl", format="rdfxml")
print("Guardada como GameOntology_ampliada.owl")

Guardada como GameOntology_ampliada.owl


# Ejercicios propuestos

### Ejercicio 1

Define la clase `JugadorConLogros` mediante la siguiente condición necesaria y suficiente:
**JugadorConLogros ≡ Player ⊓ ∃ hasAchievement.Achievement**
Crea un jugador, un logro y establece la relación correspondiente entre ambos.
1. Consulta los tipos del jugador antes de ejecutar el razonador.
2. Ejecuta Pellet.
3. Consulta de nuevo sus tipos.
4. Explica qué nueva información ha inferido el razonador y qué axiomas han permitido obtenerla.


### Ejercicio 2

Crea una nueva propiedad `recommendsGame` cuyo dominio sea `Player` y cuyo rango sea `Game`.
Crea dos individuos `persona1` y `elemento1` sin indicar explícitamente que pertenecen a `Player` y `Game`.
Añade:
**recommendsGame(persona1, elemento1)**
Ejecuta el razonador y estudia qué tipos pueden inferirse para ambos individuos.
Explica por qué se producen estas inferencias.


### Ejercicio 3

Define una clase:
**JugadorEspecializado ≡ Player ⊓ ∀ playsGame.Game**
Crea un jugador y establece que juega a un individuo de la clase `Game`.
¿Es suficiente esta información para que el razonador clasifique al jugador como `JugadorEspecializado`?
Ejecuta el razonador, observa el resultado y explica cómo influye la **Open World Assumption (OWA)**.


### Ejercicio 4

Define:
**JugadorConVariosJuegos ≡ Player ⊓ ≥ 2 playsGame.Game**
Crea un jugador y dos juegos diferentes y establece que el jugador juega a ambos.
Ejecuta el razonador y comprueba si el jugador es clasificado como `JugadorConVariosJuegos`.
Si no se obtiene inicialmente el resultado esperado, indica explícitamente que los dos juegos son individuos diferentes y vuelve a ejecutar el razonador.
Explica por qué esta información puede ser necesaria en OWL.


### Ejercicio 5

Crea una propiedad `hasMainCharacter` entre `Game` y `Character` y declárala como funcional.
Crea:
- un juego `juego1`;
- dos personajes `personaje1` y `personaje2`.
Establece:
**hasMainCharacter(juego1, personaje1)**
**hasMainCharacter(juego1, personaje2)**
Antes de ejecutar Pellet, responde:
1. ¿Es necesariamente inconsistente la ontología?
2. ¿Qué relación puede inferir el razonador entre `personaje1` y `personaje2`?
Ejecuta el razonador y estudia el resultado.
A continuación, declara explícitamente que `personaje1` y `personaje2` son diferentes y vuelve a ejecutar Pellet.
¿Qué ocurre ahora?


### Ejercicio 6

Crea una propiedad `defeated` entre jugadores y declárala como asimétrica.
Crea `jugadorA` y `jugadorB` y añade:
**defeated(jugadorA, jugadorB)**
Ejecuta el razonador y comprueba que la ontología es consistente.
A continuación añade:
**defeated(jugadorB, jugadorA)**
Ejecuta nuevamente el razonador.
¿Sigue siendo consistente la ontología? Justifica el resultado utilizando la definición de propiedad asimétrica.


### Ejercicio 7

Crea una propiedad `defeats` entre jugadores y declárala como irreflexiva.
Crea un jugador `jugadorA` y establece:
**defeats(jugadorA, jugadorA)**
Antes de ejecutar Pellet, predice si la ontología será consistente.
Ejecuta el razonador y comprueba tu respuesta.
Explica el resultado utilizando la definición de propiedad irreflexiva.


### Ejercicio 8

Crea una propiedad `isCompatibleWith` entre jugadores y declárala como reflexiva.
Crea dos jugadores: `jugadorA` y `jugadorB`
No añadas ninguna relación `isCompatibleWith` explícitamente.
Ejecuta el razonador y estudia qué relaciones deberían cumplirse como consecuencia de la reflexividad.
Comprueba en particular si:
**isCompatibleWith(jugadorA, jugadorA)**
**isCompatibleWith(jugadorB, jugadorB)**
Explica la diferencia entre estas relaciones inferidas y las relaciones introducidas explícitamente en la ontología.